# MD-GRTN v26 — PEMS08
**Target:** MAE < 13.114 | RMSE < 22.623 | MAPE < 8.471%
**v25 result:** Plateau at MAE≈15.0, GPU disconnected at epoch 85

## Fixes vs v25
1. **GAT → GCN**: v25 GAT = 226 MB per layer × 3 = 677 MB just for attention maps (caused OOM). GCN = `A@X@W` = O(N×d) memory, no N² matrix
2. **Removed hard residual**: `out + last_obs` forced model to learn noisy deltas — that's why it plateaued at MAE≈15
3. **GRU decoder** replaces flat `Linear(d*S, P)`: proper seq-to-seq with sequential inductive bias
4. **seq_len=12**: paper's value (was 16)
5. **MAE loss**: directly optimises the metric


In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed} ✓')

set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
class Config:
    data_path = "/kaggle/input/pems08/PEMS08.npz"
    adj_path  = "/kaggle/input/pems08/PEMS08.csv"
    best_path = "mdgrtn_v26_best.pt"

    num_nodes   = 170
    in_features = 3       # flow + occupancy + speed
    seq_len     = 12      # paper value
    pred_len    = 12
    feature_idx = 0
    train_ratio = 0.7
    val_ratio   = 0.1

    d_model    = 64       # GCN is param-efficient
    n_heads    = 4
    gcn_layers = 3
    st_layers  = 3
    dropout    = 0.15

    batch_size   = 32     # OOM-safe
    lr           = 1e-3
    epochs       = 200
    patience     = 30
    weight_decay = 1e-3

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config | d={cfg.d_model} | GCN={cfg.gcn_layers} | ST={cfg.st_layers} | '
      f'seq={cfg.seq_len} | batch={cfg.batch_size} | {device}')

In [ ]:
def build_adj_from_csv(csv_path, num_nodes):
    try:
        df = pd.read_csv(csv_path, header=None, skiprows=1)
        df.columns = ['from', 'to', 'cost']
        df['from'] = df['from'].astype(int)
        df['to']   = df['to'].astype(int)
        df['cost'] = df['cost'].astype(float)
        A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
        for _, row in df.iterrows():
            i, j, c = int(row['from']), int(row['to']), float(row['cost'])
            A[i, j] = c
            A[j, i] = c
        sigma   = df['cost'].std()
        A_gauss = np.exp(-(A ** 2) / (sigma ** 2 + 1e-8))
        np.fill_diagonal(A_gauss, 0.0)
        A_norm  = A_gauss / (A_gauss.sum(1, keepdims=True) + 1e-8)
        print(f'Adjacency | edges={len(df)} | nnz(>0.01)={(A_gauss > 0.01).sum()} | sigma={sigma:.1f}')
        return A_norm
    except Exception as e:
        print(f'WARNING: CSV adj failed ({e}), identity fallback')
        return np.eye(num_nodes, dtype=np.float32)


def load_pems08(cfg):
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Data shape: {data.shape}')
    mean_np   = data.mean(axis=0)
    std_np    = data.std(axis=0) + 1e-8
    data_norm = (data - mean_np[None]) / std_np[None]
    A_dist    = build_adj_from_csv(cfg.adj_path, N)
    return data_norm, mean_np, std_np, A_dist


class TrafficDataset(Dataset):
    def __init__(self, data_norm, seq_len, pred_len, feature_idx,
                 split_start=0, split_end=None):
        self.data     = data_norm
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.feat_idx = feature_idx
        T   = len(data_norm)
        end = split_end if split_end is not None else T
        self.indices = list(range(split_start, end - seq_len - pred_len + 1))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        x = self.data[i : i + self.seq_len].copy()
        y = self.data[i + self.seq_len : i + self.seq_len + self.pred_len,
                      :, self.feat_idx].copy()
        return torch.from_numpy(x), torch.from_numpy(y)


def build_dataloaders(cfg):
    set_seed()
    data_norm, mean_np, std_np, A_dist = load_pems08(cfg)
    T  = len(data_norm)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data_norm=data_norm, seq_len=cfg.seq_len,
                 pred_len=cfg.pred_len, feature_idx=cfg.feature_idx)
    dl_tr = DataLoader(TrafficDataset(**ds_kw, split_start=0,  split_end=t1), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset(**ds_kw, split_start=t1, split_end=t2), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset(**ds_kw, split_start=t2, split_end=T),  shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist

print('Data utilities ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# INPUT PROJECTION
# ══════════════════════════════════════════════════════════════════════
class InputProjection(nn.Module):
    def __init__(self, in_dim, d_model, dropout=0.1):
        super().__init__()
        self.proj  = nn.Linear(in_dim, d_model)
        self.norm  = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        h = self.drop(self.proj(x))
        h = self.norm(h)
        return self.norm2(h + self.ff(h))


# ══════════════════════════════════════════════════════════════════════
# GCN LAYER  — replaces GAT (paper uses GCN, not GAT)
# Memory: O(N*d) not O(N^2)  — eliminates OOM
# ══════════════════════════════════════════════════════════════════════
class GCNLayer(nn.Module):
    """X' = ReLU(A @ X @ W) + residual. (B, N, d) → (B, N, d)"""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.W     = nn.Linear(d_model, d_model, bias=False)
        self.norm  = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_model * 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight)
    def forward(self, x, A):
        gcn = F.relu(A @ self.W(x))
        x   = self.norm(x + self.drop(gcn))
        x   = self.norm2(x + self.drop(self.ff(x)))
        return x


# ══════════════════════════════════════════════════════════════════════
# GCN + GRU BLOCK  (paper MGRC: GCN spatial + GRU temporal)
# ══════════════════════════════════════════════════════════════════════
class GCNGRUBlock(nn.Module):
    """GCN at each timestep, then GRU across S timesteps per node."""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.gcn   = GCNLayer(d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.gru   = nn.GRU(d_model, d_model, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x, A):
        B, S, N, d = x.shape
        gcn_out = self.gcn(x.reshape(B * S, N, d), A).reshape(B, S, N, d)
        x = self.norm1(x + self.drop(gcn_out))
        x_t        = x.permute(0, 2, 1, 3).reshape(B * N, S, d)
        gru_out, _ = self.gru(x_t)
        gru_out    = gru_out.reshape(B, N, S, d).permute(0, 2, 1, 3)
        x = self.norm2(x + self.drop(gru_out))
        return x


# ══════════════════════════════════════════════════════════════════════
# MGRC MODULE  (dynamic + static adj fusion)
# ══════════════════════════════════════════════════════════════════════
class MGRCModule(nn.Module):
    def __init__(self, d_model, num_nodes, num_layers=3, dropout=0.1):
        super().__init__()
        emb_dim    = max(d_model // 4, 16)
        self.E1    = nn.Parameter(torch.randn(num_nodes, emb_dim) * 0.01)
        self.E2    = nn.Parameter(torch.randn(num_nodes, emb_dim) * 0.01)
        self.alpha = nn.Parameter(torch.zeros(1))
        self.layers = nn.ModuleList([GCNGRUBlock(d_model, dropout) for _ in range(num_layers)])
    def get_fused_adj(self, A_dist):
        A_dyna = torch.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        a      = torch.sigmoid(self.alpha)
        A_F    = a * A_dist + (1.0 - a) * A_dyna
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)
    def forward(self, x, A_dist):
        A_F = self.get_fused_adj(A_dist)
        for layer in self.layers:
            x = layer(x, A_F)
        return x


# ══════════════════════════════════════════════════════════════════════
# SPATIAL TRANSFORMER  (global cross-node attention)
# ══════════════════════════════════════════════════════════════════════
class SpatialTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(d_model * 4, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x):
        B, S, N, d = x.shape
        xf    = x.reshape(B * S, N, d)
        h, _  = self.attn(xf, xf, xf)
        xf    = self.norm1(xf + self.drop(h))
        xf    = self.norm2(xf + self.drop(self.ff(xf)))
        return xf.reshape(B, S, N, d)


# ══════════════════════════════════════════════════════════════════════
# TEMPORAL TRANSFORMER  (sinusoidal PE + learnable TPE)
# ══════════════════════════════════════════════════════════════════════
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                        (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TemporalTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout, seq_len, num_nodes):
        super().__init__()
        self.sin_pe = SinusoidalPE(d_model)
        self.W_hour = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        self.W_day  = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        self.W_week = nn.Parameter(torch.randn(num_nodes, 1) * 0.02)
        t = torch.arange(seq_len).float()
        self.register_buffer('E_hour', (t % 12 + 1).unsqueeze(0))
        self.register_buffer('E_day',  (t % 24 + 1).unsqueeze(0))
        self.register_buffer('E_week', (t % 7  + 1).unsqueeze(0))
        self.tpe_proj = nn.Linear(1, d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.GELU(),
                                   nn.Linear(d_model * 4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x):
        B, S, N, d = x.shape
        enc = (self.W_hour @ self.E_hour + self.W_day @ self.E_day + self.W_week @ self.E_week)
        enc = self.tpe_proj(enc.T.unsqueeze(0).unsqueeze(-1))
        x   = x + enc
        f   = x.permute(0, 2, 1, 3).reshape(B * N, S, d)
        f   = self.sin_pe(f)
        h, _ = self.attn(f, f, f)
        h   = self.norm1(f + self.drop(h))
        h   = self.norm2(h + self.drop(self.ff(h)))
        return h.reshape(B, N, S, d).permute(0, 2, 1, 3)


class STFormerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout, seq_len, num_nodes):
        super().__init__()
        self.spatial  = SpatialTransformerLayer(d_model, n_heads, dropout)
        self.temporal = TemporalTransformerLayer(d_model, n_heads, dropout, seq_len, num_nodes)
    def forward(self, x):
        return self.temporal(self.spatial(x))


# ══════════════════════════════════════════════════════════════════════
# FULL MODEL
# ══════════════════════════════════════════════════════════════════════
class MDGRTNv26(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        h, dr, S, P = cfg.n_heads, cfg.dropout, cfg.seq_len, cfg.pred_len
        self.input_proj = InputProjection(F, d, dr)
        self.mgrc       = MGRCModule(d, N, num_layers=cfg.gcn_layers, dropout=dr)
        self.stformer   = nn.ModuleList([
            STFormerBlock(d, h, dr, S, N) for _ in range(cfg.st_layers)])
        # GRU decoder: encode sequence, use final hidden state h_T
        self.dec_gru  = nn.GRU(d, d, batch_first=True)
        self.out_proj = nn.Linear(d, P)

    def forward(self, x_rec, A_dist):
        B, S, N, _ = x_rec.shape
        x = self.input_proj(x_rec)    # (B, S, N, d)
        x = self.mgrc(x, A_dist)      # (B, S, N, d)
        for blk in self.stformer:
            x = blk(x)
        # GRU decoder
        x_dec  = x.permute(0, 2, 1, 3).reshape(B * N, S, -1)  # (B*N, S, d)
        _, h_T = self.dec_gru(x_dec)                            # h_T: (1, B*N, d)
        out    = self.out_proj(h_T.squeeze(0))                  # (B*N, P)
        return out.reshape(B, N, -1).permute(0, 2, 1)          # (B, P, N)


set_seed()
model = MDGRTNv26(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MDGRTNv26 | Parameters: {total:,}')
print(f'  d={cfg.d_model} | GCN={cfg.gcn_layers} | ST={cfg.st_layers}')
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    out   = model(dummy, torch.eye(cfg.num_nodes).to(device))
print(f'Output shape: {out.shape}  ✓  (expect [2, 12, 170])')
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    with torch.amp.autocast('cuda'):
        _ = model(dummy, torch.eye(cfg.num_nodes).to(device))
    print(f'VRAM (batch=2): {torch.cuda.memory_allocated()/1024**2:.0f} MB')

In [ ]:
def masked_mae_loss(pred, true):
    """MAE loss on normalised values. Masks near-zero (sensor-off) positions."""
    mask = (true.abs() > 0.1).float()
    return (torch.abs(pred - true) * mask).sum() / (mask.sum() + 1e-8)

def masked_mae(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return (torch.abs(pred - true) * mask).sum() / (mask.sum() + 1e-8)

def masked_rmse(pred, true, null_val=0.0):
    mask = (true != null_val).float()
    return torch.sqrt(((pred - true) ** 2 * mask).sum() / (mask.sum() + 1e-8))

def masked_mape(pred, true, low_thresh=10.0):
    mask = (true.abs() > low_thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred - true) / (true.abs() + 1.0)) * mask).sum() / mask.sum() * 100

print('Metrics defined.')

In [ ]:
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, optimizer, scheduler, A_dist, device):
    model.train()
    total = 0.0
    for x_rec, y in loader:
        x_rec = x_rec.to(device)
        y     = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_dist)
            loss = masked_mae_loss(pred, y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, y in loader:
        x_rec = x_rec.to(device)
        y     = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_dist)
        pred_d = pred.float() * std_flow[None, None, :] + mean_flow[None, None, :]
        y_d    = y.float()    * std_flow[None, None, :] + mean_flow[None, None, :]
        maes.append(masked_mae(pred_d, y_d).item())
        rmses.append(masked_rmse(pred_d, y_d).item())
        mapes.append(masked_mape(pred_d, y_d).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

def save_best(model, path):
    torch.save(model.state_dict(), path)
    print(f'  Best saved → {path}')

print('Train/eval ready.')

In [ ]:
set_seed()
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_dist_np).to(device)
print(f'Train batches: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}')

In [ ]:
set_seed()
model = MDGRTNv26(cfg).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=cfg.lr, epochs=cfg.epochs,
    steps_per_epoch=len(dl_train), pct_start=0.10,
    anneal_strategy='cos', div_factor=10.0, final_div_factor=500.0)

best_val_mae = float('inf')
patience_cnt = 0
history = {'train_loss': [], 'val_mae': [], 'val_rmse': [], 'val_mape': []}

print(f'MDGRTNv26 | d={cfg.d_model} | GCN={cfg.gcn_layers} | ST={cfg.st_layers}')
print(f'lr={cfg.lr} | wd={cfg.weight_decay} | dropout={cfg.dropout} | batch={cfg.batch_size}')
print(f'Baseline → MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('=' * 65)

for epoch in range(1, cfg.epochs + 1):
    train_loss = train_epoch(model, dl_train, optimizer, scheduler, A_dist, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, A_dist, device, mean_flow, std_flow)

    history['train_loss'].append(train_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_cnt = 0
        save_best(model, cfg.best_path)
        tag = '  ← best ✓'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stopping at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    if epoch % 5 == 0 or tag:
        print(f'Ep {epoch:03d} | Loss={train_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}')

print(f'\nBest Val MAE: {best_val_mae:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], color='steelblue', label='Train Loss')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout(); plt.savefig('training_curves_v26.png', dpi=150); plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_style_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, y in loader:
        x_rec = x_rec.to(device); y = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_dist)
        pred_d = pred.float() * std_flow[None, None, :] + mean_flow[None, None, :]
        y_d    = y.float()    * std_flow[None, None, :] + mean_flow[None, None, :]
        all_pred.append(pred_d.cpu()); all_true.append(y_d.cpu())
    P = torch.cat(all_pred, dim=0); Y = torch.cat(all_true, dim=0)
    mae  = torch.abs(P - Y).mean().item()
    rmse = ((P - Y) ** 2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask] - Y[mask]) / (Y[mask].abs() + 1.0))).mean().item() * 100
    print('\n' + '=' * 60)
    print('  TEST RESULTS  —  MDGRTNv26  —  all 12 steps')
    print('=' * 60)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Δ={mae - 13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Δ={rmse - 22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Δ={mape - 8.471:+.3f}%')
    print('=' * 60)
    return mae, rmse, mape

mae, rmse, mape = paper_style_eval(model, dl_test, A_dist, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A_dist, device, mean_flow, std_flow):
    model.eval()
    buf = {h: {'mae': [], 'rmse': [], 'mape': []} for h in [2, 5, 11]}
    for x_rec, y in loader:
        x_rec = x_rec.to(device); y = y.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, A_dist)
        pred_d = pred.float() * std_flow[None, None, :] + mean_flow[None, None, :]
        y_d    = y.float()    * std_flow[None, None, :] + mean_flow[None, None, :]
        for h in buf:
            buf[h]['mae'].append(masked_mae(pred_d[:, h, :], y_d[:, h, :]).item())
            buf[h]['rmse'].append(masked_rmse(pred_d[:, h, :], y_d[:, h, :]).item())
            buf[h]['mape'].append(masked_mape(pred_d[:, h, :], y_d[:, h, :]).item())
    print(f"{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}")
    print('-' * 50)
    for h, lbl in zip([2, 5, 11], ['3-step (15min)', '6-step (30min)', '12-step (60min)']):
        m = {k: np.mean(v) for k, v in buf[h].items()}
        print(f"{lbl:>14} | {m['mae']:>8.3f} | {m['rmse']:>8.3f} | {m['mape']:>8.2f}%")

horizon_eval(model, dl_test, A_dist, device, mean_flow, std_flow)